# Dreamer-FuN: Hierarchical World Models for Long-Horizon Sample-Efficient Exploration

This notebook provides a complete workflow for setting up, training, and evaluating the Dreamer-FuN agent. It includes configuration loading, environment initialization, agent setup, a training loop, evaluation, and extensive visualization code to generate publication-quality figures.


In [ ]:
import torch
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import deque

# Import modules from src
from src.config import (
    WorldModelConfig,
    ManagerConfig,
    WorkerConfig,
    TrainingConfig,
    EnvironmentConfig,
)
from src.agents import DreamerFuNAgent
from src.data import ReplayBuffer, DreamerFuNDataset, create_dataloader
from src.environments import DreamerFuNEnvWrapper, make_env
from src.utils import set_seed, get_device, Logger, save_checkpoint, load_checkpoint

# Configure plotting style
sns.set_theme()
plt.style.use("ggplot")

# Create pictures directory if it doesn't exist
if not os.path.exists("../pictures"):
    os.makedirs("../pictures")

print("Libraries and modules imported successfully.")
print(f"Using device: {get_device()}")

In [ ]:
# 2. Configuration Loading and Environment Initialization

# Load configurations
world_model_config = WorldModelConfig(
    observation_shape=(3, 64, 64),  # Example: (C, H, W) for image observations
    latent_dim=32,
    hidden_dim=256,
    action_dim=None,  # Will be set by environment
    feature_dim=1024,
    rssm_layers=1,
    rssm_type="discrete",
    kl_loss_scale=0.1,
    reward_clip=1.0,
    free_nats=0.0,
)
manager_config = ManagerConfig(
    goal_dim=32, goal_horizon=10, learning_rate=1e-4, discount_factor=0.99
)
worker_config = WorkerConfig(
    sub_goal_horizon=5,
    intrinsic_reward_weight=0.5,
    extrinsic_reward_weight=0.5,
    learning_rate=1e-4,
    discount_factor=0.99,
    exploration_amount=0.3,
)
training_config = TrainingConfig(
    batch_size=50,
    sequence_length=50,
    replay_buffer_size=1_000_000,
    environment_steps=100_000,  # Reduced for demo
    model_retain_steps=1_000_000,
    train_every_steps=100,
    seed=42,
    log_interval=100,
    eval_interval=5_000,
    checkpoint_interval=25_000,
    gradient_clip_norm=100.0,
)
environment_config = EnvironmentConfig(
    env_name="CartPole-v1",  # Using CartPole for simplicity initially
    action_repeat=1,
    image_size=(64, 64),
    grayscale=False,
    reward_scale=1.0,
    time_limit=500,
)

# Set seed for reproducibility
set_seed(training_config.seed)
device = get_device()

# Create environment
# For CartPole, observations are typically vectors, so we adjust observation_shape
if environment_config.env_name == "CartPole-v1":
    dummy_env = gym.make(environment_config.env_name)
    obs_space_shape = dummy_env.observation_space.shape
    action_space_shape = (
        (dummy_env.action_space.n,)
        if isinstance(dummy_env.action_space, gym.spaces.Discrete)
        else dummy_env.action_space.shape
    )
    dummy_env.close()

    world_model_config.observation_shape = obs_space_shape
    world_model_config.action_dim = action_space_shape[0]

    env = make_env(
        environment_config.env_name, environment_config, training_config.seed
    )
    print(f"Environment: {environment_config.env_name}")
    print(f"Observation Space: {env.observation_space.shape}")
    print(f"Action Space: {env.action_space.shape}")

else:
    # For image-based environments, you would ensure observation_shape is (C, H, W)
    # and action_dim is set correctly.
    # For now, keeping CartPole specific logic.
    env = make_env(
        environment_config.env_name, environment_config, training_config.seed
    )
    print(f"Environment: {environment_config.env_name}")
    print(f"Observation Space: {env.observation_space.shape}")
    print(f"Action Space: {env.action_space.shape}")

world_model_config.action_dim = (
    env.action_space.shape[0] if isinstance(env.action_space, gym.spaces.Box) else 1
)  # Assuming 1 for discrete action ID

# Initialize logger
logger = Logger(experiment_name="dreamer_fun", log_dir="../logs")
print(f"Logging to: {logger.log_dir}")

In [ ]:
# 3. Agent and Replay Buffer Initialization

agent = DreamerFuNAgent(
    world_model_config=world_model_config,
    manager_config=manager_config,
    worker_config=worker_config,
    training_config=training_config,
    action_dim=world_model_config.action_dim,
    device=device,
)

replay_buffer = agent.replay_buffer  # Agent initializes its own replay buffer

print("DreamerFuNAgent and ReplayBuffer initialized successfully.")

In [ ]:
# 4. Training Loop

# Placeholder for training loop. No actual training execution for this static notebook.
# The actual training would involve interacting with the environment,
# storing transitions, and then iteratively updating the world model, manager, and worker.

print("Starting training simulation (no actual execution of full training loop).")

episode_returns = deque(maxlen=100)  # To track recent episode returns
obs, info = env.reset()
current_episode_reward = 0
episode_step_count = 0

for step in range(1, training_config.environment_steps + 1):
    action = agent.act(obs, training=True)
    next_obs, reward, done, info = env.step(action)  # info can be dictionary or None

    # Store transition in replay buffer
    agent.store_transition(obs, action, reward, done, next_obs)

    current_episode_reward += reward
    episode_step_count += 1

    obs = next_obs

    if done or episode_step_count >= environment_config.time_limit:
        episode_returns.append(current_episode_reward)
        logger.log(step, {"episode_reward": current_episode_reward})
        avg_episode_reward = np.mean(episode_returns) if len(episode_returns) > 0 else 0
        logger.log(step, {"avg_episode_reward": avg_episode_reward})

        print(
            f"Step {step}: Episode finished with reward {current_episode_reward:.2f}. Avg reward: {avg_episode_reward:.2f}"
        )
        obs, info = env.reset()
        agent.reset_episode_state()
        current_episode_reward = 0
        episode_step_count = 0

    if (
        step % training_config.train_every_steps == 0
        and replay_buffer.size > training_config.sequence_length
    ):
        # Sample a batch of sequences from the replay buffer
        batch = replay_buffer.sample(
            training_config.batch_size, training_config.sequence_length
        )

        # Update World Model
        wm_losses = agent.update_world_model(batch)
        logger.log(step, {f"world_model_{k}": v for k, v in wm_losses.items()})

        # Update Manager and Worker
        agent_losses = agent.update_manager_and_worker(batch)
        logger.log(step, {f"agent_{k}": v for k, v in agent_losses.items()})

    if step % training_config.eval_interval == 0:
        print(f"\n--- Evaluating at step {step} ---")
        # Placeholder for actual evaluation logic
        # avg_eval_reward = evaluate_agent(env, agent, num_episodes=5)
        # logger.log(step, {'avg_eval_reward': avg_eval_reward})
        print("Evaluation placeholder executed.")

    if step % training_config.checkpoint_interval == 0:
        save_checkpoint(
            agent.world_model,
            agent.world_model_optimizer,
            step,
            logger.log_dir,
            "world_model",
        )
        save_checkpoint(
            agent.manager_actor,
            agent.manager_actor_optimizer,
            step,
            logger.log_dir,
            "manager_actor",
        )
        save_checkpoint(
            agent.manager_critic,
            agent.manager_critic_optimizer,
            step,
            logger.log_dir,
            "manager_critic",
        )
        save_checkpoint(
            agent.worker_actor,
            agent.worker_actor_optimizer,
            step,
            logger.log_dir,
            "worker_actor",
        )
        save_checkpoint(
            agent.worker_critic,
            agent.worker_critic_optimizer,
            step,
            logger.log_dir,
            "worker_critic",
        )

print("Training simulation complete.")
logger.close()
env.close()

In [ ]:
# 5. Visualization and Analysis

# This section provides code to generate publication-quality figures.
# The plots are generated from the logged data and saved to the '../pictures' directory.

print("Generating visualizations...")

# --- Plot 1: World Model Losses ---
plt.figure(figsize=(12, 6))
plt.plot(
    [s for s, v in logger.get_metric_history("world_model_obs_loss")],
    [v for s, v in logger.get_metric_history("world_model_obs_loss")],
    label="Observation Reconstruction Loss",
)
plt.plot(
    [s for s, v in logger.get_metric_history("world_model_reward_loss")],
    [v for s, v in logger.get_metric_history("world_model_reward_loss")],
    label="Reward Prediction Loss",
)
plt.plot(
    [s for s, v in logger.get_metric_history("world_model_kl_loss")],
    [v for s, v in logger.get_metric_history("world_model_kl_loss")],
    label="KL Divergence Loss",
)
plt.plot(
    [s for s, v in logger.get_metric_history("world_model_total_wm_loss")],
    [v for s, v in logger.get_metric_history("world_model_total_wm_loss")],
    label="Total World Model Loss",
    linewidth=2,
    color="black",
)
plt.xlabel("Environment Steps")
plt.ylabel("Loss")
plt.title("World Model Training Losses")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("../pictures/fig_01_world_model_losses.png", dpi=300)
plt.close()
print("Saved fig_01_world_model_losses.png")

# --- Plot 2: Agent Policy Losses (Manager and Worker) ---
plt.figure(figsize=(12, 6))
plt.plot(
    [s for s, v in logger.get_metric_history("agent_manager_total_loss")],
    [v for s, v in logger.get_metric_history("agent_manager_total_loss")],
    label="Manager Total Loss",
)
plt.plot(
    [s for s, v in logger.get_metric_history("agent_worker_total_loss")],
    [v for s, v in logger.get_metric_history("agent_worker_total_loss")],
    label="Worker Total Loss",
)
plt.xlabel("Environment Steps")
plt.ylabel("Loss")
plt.title("Manager and Worker Policy Training Losses")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("../pictures/fig_02_agent_policy_losses.png", dpi=300)
plt.close()
print("Saved fig_02_agent_policy_losses.png")

# --- Plot 3: Episode Rewards (Average) ---
plt.figure(figsize=(12, 6))
plt.plot(
    [s for s, v in logger.get_metric_history("avg_episode_reward")],
    [v for s, v in logger.get_metric_history("avg_episode_reward")],
    label="Average Episode Reward",
    color="green",
)
plt.xlabel("Environment Steps")
plt.ylabel("Average Reward")
plt.title(f"Average Episode Reward over Training ({environment_config.env_name})")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("../pictures/fig_03_average_episode_reward.png", dpi=300)
plt.close()
print("Saved fig_03_average_episode_reward.png")

# --- Plot 4: Intrinsic Rewards ---
plt.figure(figsize=(12, 6))
plt.plot(
    [s for s, v in logger.get_metric_history("agent_manager_intrinsic_reward")],
    [v for s, v in logger.get_metric_history("agent_manager_intrinsic_reward")],
    label="Manager Intrinsic Reward",
)
plt.plot(
    [s for s, v in logger.get_metric_history("agent_worker_intrinsic_reward")],
    [v for s, v in logger.get_metric_history("agent_worker_intrinsic_reward")],
    label="Worker Intrinsic Reward",
)
plt.xlabel("Environment Steps")
plt.ylabel("Reward Value")
plt.title("Intrinsic Rewards for Manager and Worker")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("../pictures/fig_04_intrinsic_rewards.png", dpi=300)
plt.close()
print("Saved fig_04_intrinsic_rewards.png")

# --- Plot 5: Latent Space Visualization (Placeholder) ---
# This would require running a trained agent and collecting latent states,
# then projecting them (e.g., with PCA or t-SNE) for visualization.
# For a static notebook, this is a conceptual placeholder.

print(
    "Latent space visualization is a placeholder and would require active agent execution."
)

# Example for a conceptual latent space plot:
# plt.figure(figsize=(10, 8))
# # Assume you have 'latent_states' and 'goals' collected during rollout
# # from src.utils import reduce_dimension_for_plot
# # reduced_latents = reduce_dimension_for_plot(latent_states)
# # reduced_goals = reduce_dimension_for_plot(goals)
# # plt.scatter(reduced_latents[:, 0], reduced_latents[:, 1], c='blue', label='Latent States', alpha=0.5)
# # plt.scatter(reduced_goals[:, 0], reduced_goals[:, 1], c='red', marker='X', s=100, label='Manager Goals')
# plt.title('Conceptual Latent Space Visualization (Placeholder)')
# plt.xlabel('Dimension 1')
# plt.ylabel('Dimension 2')
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.savefig('../pictures/fig_05_latent_space.png', dpi=300)
# plt.close()

# --- Plot 6: Reconstructed Observations (Placeholder) ---
# This would involve feeding actual observations through the world model's decoder
# and visualizing the original vs. reconstructed images.

print(
    "Reconstructed observations visualization is a placeholder and would require active agent execution."
)

# Example for a conceptual reconstructed observations plot:
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# # Assume original_obs and reconstructed_obs are available
# # axes[0].imshow(original_obs.squeeze().cpu().numpy().transpose(1, 2, 0)) # For (C, H, W) images
# # axes[0].set_title('Original Observation')
# # axes[0].axis('off')
# # axes[1].imshow(reconstructed_obs.squeeze().cpu().numpy().transpose(1, 2, 0)) # For (C, H, W) images
# # axes[1].set_title('Reconstructed Observation')
# # axes[1].axis('off')
# plt.suptitle('Conceptual Original vs. Reconstructed Observations (Placeholder)')
# plt.tight_layout()
# plt.savefig('../pictures/fig_06_reconstructed_observations.png', dpi=300)
# plt.close()

print(
    "All visualization code added. You can run this notebook to generate the plots after training."
)

# Dreamer-FuN: Hierarchical World Models for Long-Horizon Sample-Efficient Exploration

This notebook provides a complete workflow for setting up, training, and evaluating the Dreamer-FuN agent. It includes configuration loading, environment initialization, agent setup, a training loop, evaluation, and extensive visualization code to generate publication-quality figures.
